# domain-audit: Comprehensive Domain Health Scanner

Audit any domain for DNS records, subdomains, SSL/TLS health, HTTP security headers, WHOIS info, open ports, email security (SPF/DKIM/DMARC), and technology detection.

Results are graded A-F with actionable fix recommendations.

**No API keys required. Just enter a domain and run.**

---

> **Legal disclaimer:** Only scan domains you own or have explicit permission to test. Unauthorized scanning may violate computer fraud laws in your jurisdiction. This tool performs active network probes (port scanning, DNS queries, HTTP requests) against the target domain. By using this tool, you accept full responsibility for ensuring you have authorization to scan the target.

In [ ]:
# Step 1: Install domain-audit
# Uses --no-deps to avoid conflicts with Colab's pinned packages (requests, cryptography).
# Colab already has all required dependencies pre-installed.

# Option 1: Install from PyPI (recommended, once published)
# !pip install domain-audit --no-deps -q

# Option 2: Install latest from GitHub (force fresh copy)
import subprocess, sys
subprocess.check_call([
    sys.executable, "-m", "pip", "install",
    "--force-reinstall", "--no-cache-dir", "--no-deps",
    "git+https://github.com/Opticsponge/domain-audit.git",
    "-q",
])

# Option 3: Install from YOUR GitHub fork (replace YOUR_USERNAME)
# !pip install --force-reinstall --no-cache-dir --no-deps git+https://github.com/YOUR_USERNAME/domain-audit.git -q

# Option 4: Install a specific branch from your fork
# !pip install --force-reinstall --no-cache-dir --no-deps git+https://github.com/YOUR_USERNAME/domain-audit.git@feature/my-branch -q

# Option 5: Clone and install in editable mode (for live editing)
# !git clone https://github.com/YOUR_USERNAME/domain-audit.git /content/domain-audit
# !pip install --no-deps -e /content/domain-audit -q

# Flush cached modules so the fresh install is picked up
import sys as _sys
for _mod in list(_sys.modules):
    if _mod.startswith("domain_audit"):
        del _sys.modules[_mod]
print("✅ domain-audit installed")

In [ ]:
# Step 2: Run the audit
# Change the domain below to scan any domain you want

DOMAIN = "google.com"  # <-- Change this to your domain

# Grade filter: pick which grades to show in the report
# Options: "A", "B", "C", "F", or "all" (default: show everything)
GRADE_FILTER = "all"  # <-- Change to "F" to see only failures, "C" for warnings, etc.

# Scanner selection: which modules to run
# Available scanners and what they do:
#   dns        - DNS records (A, AAAA, MX, NS, TXT, CNAME, SOA, SRV, CAA) + zone transfer test
#   subdomains - Subdomain discovery via Certificate Transparency logs + per-subdomain probes
#   ssl        - SSL/TLS certificate validity, expiration, hostname match, protocol version
#   headers    - HTTP security headers (HSTS, CSP, etc.) + redirect chain + cookie security
#   whois      - WHOIS/RDAP registration info, domain age, expiry, DNSSEC, transfer locks
#   ports      - Port scan (16 ports: HTTP, SSH, SMTP, databases, RDP, Elasticsearch, MongoDB)
#   email      - Email security: SPF (with lookup chain), DKIM (36 selectors), DMARC policy
#   tech       - Technology fingerprinting via headers, meta tags, and URL patterns
SCANNERS = None  # <-- Set to ["ssl", "dns", "headers"] to run specific ones, or None for all

from domain_audit import audit
results = audit(DOMAIN, only=SCANNERS, show=False, deep_subdomains=True)

In [ ]:
# Step 3: View Report
# Uses the GRADE_FILTER from Step 2. Change it there and re-run both cells.
# You can also click the filter buttons at the top of the report to toggle grades.

from domain_audit.report_tables import display
display(results, grade_filter=GRADE_FILTER)

In [ ]:
# Step 4 (Optional): Export results

# Export as JSON
results.to_json("audit_results.json")
print("Saved to audit_results.json")

# Export as CSV
results.to_csv("audit_results.csv")
print("Saved to audit_results.csv")

# Download files (Colab)
try:
    from google.colab import files
    files.download("audit_results.json")
    files.download("audit_results.csv")
except ImportError:
    pass

In [ ]:
# Step 5 (Optional): Run specific scanners only

# Available scanners: dns, subdomains, ssl, headers, whois, ports, email, tech
ssl_only = audit(DOMAIN, only=["ssl", "headers"], show=False)
display(ssl_only)